### (i) Using @ tool decorator

In [1]:
from langchain_core.tools import tool

In [2]:
# Step 1 - Create a function
def multiply(a, b):
    """Multiply two numbers""" # Added doc string which tells what the function will do. It is highly recommended to use this.
    return a * b

In [3]:
# Step 2 - Add type hints
def multiply(a : int, b : int) -> int:
    """Multiply two numbers""" # It is still not compulsory but it is highly recommended to use this
    return a * b

In [5]:
# Step 3 - Add tool decorator
@tool # It is used to turn a function into a tool that the AI can see
def multiply(a : int, b : int) -> int:
    """Multiply two numbers""" 
    return a * b

In [8]:
result = multiply.invoke(
    {
        "a" : 10,
        "b" : 5
    }
)
print(result)

50


In [9]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [ ]:
print(multiply.args_schema.model_json_schema()) # This is what the LLM see when we use the tool

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


### Using StructuredTool & Pydantic (More strict way to create the tool)

In [14]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [18]:
class AddInput(BaseModel):
    a : int = Field(required = True, description = "The first number to add")
    b : int = Field(required = True, description = "The second number to add")

C:\Users\amand\AppData\Local\Temp\ipykernel_11956\3966181432.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a : int = Field(required = True, description = "The first number to add")
C:\Users\amand\AppData\Local\Temp\ipykernel_11956\3966181432.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b : int = Field(required = True, description = "The second number to add")


In [17]:
def add_func(a : int, b : int) -> int:
    return a + b

In [20]:
add_tool = StructuredTool.from_function(
    func = add_func,
    name = "addition",
    description = "Add two numbers",
    args_schema = AddInput
)

In [21]:
result = add_tool.invoke(
    {
        "a" : 10,
        "b" : 30
    }
)
result

40

In [22]:
print(add_tool.name)
print(add_tool.description)
print(add_tool.args)

addition
Add two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


### Using BaseTool class

In [23]:
from langchain_core.tools import BaseTool
from typing import Type

In [24]:
class MultiplyInput(BaseModel):
    a : int = Field(required = True, description = "First number")
    b : int = Field(required = True, description = "Second number")

C:\Users\amand\AppData\Local\Temp\ipykernel_11956\2991621374.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a : int = Field(required = True, description = "First number")
C:\Users\amand\AppData\Local\Temp\ipykernel_11956\2991621374.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b : int = Field(required = True, description = "Second number")


In [25]:
class MultiplyTool(BaseTool):
    name : str = "multiply" # name of the tool
    description : str = "Multiply two numbers"
    args_schema : Type[BaseModel] = MultiplyInput

    def _run(self, a : int, b : int) -> int:
        return a * b

In [26]:
multiply_tool = MultiplyTool()

In [27]:
result = multiply_tool.invoke(
    {
        "a" : 2,
        "b" : 890
    }
)
result

1780

In [29]:
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

multiply
Multiply two numbers
{'a': {'description': 'First number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'integer'}}
